# T33 - 信用债规模

## 第2章：环境配置与初始化

---

### 环境要求

- Python 3.8+
- MySQL 数据库
- PostgreSQL 数据库（用于ABS数据）

### 导入依赖库

In [ ]:
# 标准库
import os
import sys
import datetime
from datetime import datetime, date, timedelta, time
import logging
import warnings

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import sqlalchemy
from sqlalchemy import text, create_engine
from sqlalchemy.orm import sessionmaker
import sqlalchemy.pool as pool

# 忽略警告
warnings.filterwarnings('ignore')

print("依赖库导入成功")

### 配置参数

使用环境变量获取数据库连接配置，避免硬编码密码。

In [ ]:
# 从环境变量获取数据库配置
# 如果环境变量未设置，使用默认值（仅用于开发环境）

# MySQL配置
MYSQL_HOST = os.environ.get('MYSQL_HOST', 'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com')
MYSQL_PORT = os.environ.get('MYSQL_PORT', '3306')
MYSQL_USER = os.environ.get('MYSQL_USER', 'hz_work')
MYSQL_PASSWORD = os.environ.get('MYSQL_PASSWORD', '')  # 从环境变量获取
MYSQL_DATABASE = os.environ.get('MYSQL_DATABASE', 'bond')

# PostgreSQL配置
PG_HOST = os.environ.get('PG_HOST', '139.224.107.113')
PG_PORT = os.environ.get('PG_PORT', '18032')
PG_USER = os.environ.get('PG_USER', 'postgres')
PG_PASSWORD = os.environ.get('PG_PASSWORD', '')  # 从环境变量获取
PG_DATABASE = os.environ.get('PG_DATABASE', 'tsdb')

print("配置参数加载完成")

### 分类配置

In [ ]:
# 分类配置
CLASSIFICATION_CONFIG = {
    # 久期分段阈值
    'term_thresholds': {
        0.5: 0.75,   # 久期<0.75年 → 0.5年
        1: 1.5,      # 久期<1.5年 → 1年
        2: 2.5,      # 久期<2.5年 → 2年
        3: 3.5,      # 久期<3.5年 → 3年
        4: 4.5,      # 久期<4.5年 → 4年
        5: 6,        # 久期<6年 → 5年
        7: 8,        # 久期<8年 → 7年
        10: float('inf')  # 久期>=8年 → 10年
    },
    
    # 评级级别
    'rating_levels': ['AAA', 'AA+', 'AA', 'AA-', 'A', '其他', '无评级'],
    
    # 债券大类
    'major_types': ['城投', '产业', '金融', 'ABS'],
    
    # 金融机构子类
    'finance_subtypes': ['银行', '证券', '保险', '其他非银金融机构'],
    
    # 数据源表
    'data_sources': {
        'basicinfo': ['basicinfo_credit', 'basicinfo_finance', 'basicinfo_abs'],
        'marketinfo': ['marketinfo_credit', 'marketinfo_finance', 'marketinfo_abs']
    }
}

print("分类配置加载完成")

### 日志配置

In [ ]:
# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

def log_progress(message):
    """记录处理进度"""
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{current_time}] {message}")
    logger.info(message)

log_progress("日志系统初始化完成")

### 数据库连接

In [ ]:
def create_mysql_engine():
    """创建MySQL数据库连接引擎"""
    connection_string = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
    engine = create_engine(
        connection_string,
        poolclass=pool.NullPool,
        pool_recycle=3600,
        echo=False
    )
    return engine

def create_pg_engine():
    """创建PostgreSQL数据库连接引擎"""
    connection_string = f'postgresql://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DATABASE}'
    engine = create_engine(
        connection_string,
        pool_recycle=3600,
        echo=False
    )
    return engine

# 创建连接
sql_engine = create_mysql_engine()
log_progress("MySQL数据库连接创建成功")

# PostgreSQL连接（按需创建）
# pg_engine = create_pg_engine()
# log_progress("PostgreSQL数据库连接创建成功")

### 测试数据库连接

In [ ]:
# 测试MySQL连接
try:
    with sql_engine.connect() as conn:
        result = conn.execute(text("SELECT 1 as test"))
        print(f"MySQL连接测试成功: {result.fetchone()}")
except Exception as e:
    print(f"MySQL连接测试失败: {e}")

In [ ]:
# 检查必要的表是否存在
required_tables = [
    'basicinfo_credit',
    'basicinfo_finance', 
    'basicinfo_abs',
    'marketinfo_credit',
    'marketinfo_finance',
    'marketinfo_abs',
    'basicinfo_xzqh_ct',
    'basicinfo_industrytype1',
    'basicinfo_all'
]

try:
    with sql_engine.connect() as conn:
        for table in required_tables:
            result = conn.execute(text(f"SHOW TABLES LIKE '{table}'"))
            exists = result.fetchone() is not None
            status = "存在" if exists else "不存在"
            print(f"表 {table}: {status}")
except Exception as e:
    print(f"表检查失败: {e}")

### 辅助函数

In [ ]:
def format_dt_list(dt_list):
    """格式化日期列表为SQL IN子句格式"""
    if not dt_list:
        return None
    if len(dt_list) == 1:
        return f"('{dt_list[0]}')"
    return tuple(dt_list)

def classify_term(duration):
    """根据久期分类期限"""
    if pd.isna(duration):
        return None
    thresholds = CLASSIFICATION_CONFIG['term_thresholds']
    for label, upper in thresholds.items():
        if duration < upper:
            return label
    return 10

def standardize_rating(rating):
    """标准化评级"""
    if pd.isna(rating) or rating == '' or rating == '--':
        return '无评级'
    elif 'AAA' in str(rating):
        return 'AAA'
    elif 'AA+' in str(rating):
        return 'AA+'
    elif 'AA-' in str(rating):
        return 'AA-'
    elif 'AA' in str(rating):
        return 'AA'
    elif 'A' in str(rating):
        return 'A'
    else:
        return '其他'

print("辅助函数定义完成")

In [ ]:
# 测试辅助函数
print("久期分类测试:")
for d in [0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0]:
    print(f"  久期 {d}年 → {classify_term(d)}年")

print("\n评级标准化测试:")
for r in ['AAA', 'AA+', 'AA', 'AA-', 'A', 'BBB', '', None]:
    print(f"  评级 '{r}' → '{standardize_rating(r)}'")

---

### 环境初始化完成

**已初始化**:
- 依赖库导入
- 配置参数加载
- 日志系统配置
- 数据库连接创建
- 辅助函数定义

---

**下一章**: [3_数据采集与ETL](3_数据采集与ETL.ipynb)